# Skin Cancer Detection with Convolutional Neural Networks

In this project, we develop a deep learning model using Keras for skin cancer detection. We use the "Skin Cancer MNIST: HAM10000" dataset, which contains images of different types of skin lesions. The goal is to classify these images into several skin cancer categories.l.

# Project Start

In [ ]:
# import libraries
# for data manipulation
import pandas as pd
import numpy as np

# for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from keras.layers import Conv2D , MaxPooling2D ,Dropout , Flatten , Dense ,BatchNormalization ,Concatenate ,Input 
from keras.models import Sequential ,Model

# SK Learn library to split train and test data
from sklearn.model_selection import train_test_split

# other libraries
import cv2
import os
from PIL import Image
from tensorflow.keras.preprocessing.image import img_to_array, load_img

## Data Loading

In [ ]:
path_dir_option = ''

if path_dir_option == 'local':

    data_hmnist_path =os.path.join('archive', 'HAM10000_metadata.csv')

else:

    data_hmnist_path = os.path.join('/kaggle/input','skin-cancer-mnist-ham10000/HAM10000_metadata.csv')

print(data_hmnist_path)

data_hmnist = pd.read_csv(data_hmnist_path)

data_hmnist.head()

# Data Analysis

Let's start with an exploratory data analysis.

In [ ]:
#add column with path to image

data_hmnist['path'] = data_hmnist['image_id'].apply(lambda x: os.path.join(data_hmnist_path, 'HAM10000_images', x + '.jpg'))
data_hmnist.head()

In [ ]:
# checking for the null values
data_hmnist.isna().sum()

In [ ]:
# replacing the missing values of age with mean
data_hmnist['age'] = data_hmnist['age'].fillna(round(data_hmnist['age'].mean()))

data_hmnist.isna().sum()

After reviewing the data, we can see that there are 7 different classes of skin cancer. Each class has a different number of images.

## Statistical Analysis

In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt

# distribution of ages
plt.figure(figsize=(12,6))

sns.countplot(data=data_hmnist ,x='age',palette='viridis')

plt.title('Distribution of Age')

plt.xticks(rotation=90)

plt.show()

In [ ]:
# distribution of other features

# dx, dx-type, localization
plt.figure(figsize=(12, 6))

plt.subplot(1,3,1)
sns.countplot(data=data_hmnist , x='dx' ,palette='viridis')
plt.title('Distribution of dx')
plt.xticks(rotation=45)

plt.subplot(1,3,2)
sns.countplot(data=data_hmnist,x='localization' ,palette='viridis')
plt.title('Distribution of localization')
plt.xticks(rotation=90)

plt.subplot(1,3,3)
sns.countplot(data=data_hmnist , x = 'dx_type' , palette='viridis')
plt.title('Distribution of dx_type')

plt.tight_layout()
plt.show()

In [ ]:
# replacing unknown values in the sex column with male
# to prevent having extra column during label encoding

data_hmnist['sex'] = data_hmnist['sex'].replace('unknown', 'male')

data_hmnist['sex'].value_counts()

In [ ]:
# X_features include - age, dx_type, sex, localization

# y_feature - dx
metadata_features = data_hmnist[['age', 'dx_type', 'sex', 'localization']].copy()
metadata_features.head()

Categorical data is converted using one hot encoding.

In [ ]:
# changing categorical data into numerical data using dummies

metadata_features = pd.get_dummies(metadata_features, columns=['dx_type', 'sex', 'localization'], drop_first=True, dtype=int)
metadata_features.head()

In [ ]:
# normalizing the age value

metadata_features['age'] = metadata_features['age'] / metadata_features['age'].max()
metadata_features

Converting the 7 labels to 2 categories:

- 'Melanocytic Nevi (nv)' - Non Cancerous (0)
- 'Benign Keratosis-like Lesions (bkl)' - Non Cancerous (0)
- 'Dermatofibroma (df)' - Non Cancerous (0)
- 'Melanoma (mel)' - Cancerous (1)
- 'Vascular Lesions (vasc)' - Non Cancerous (0)
- 'Basal Cell Carcinoma (bcc)' - Cancerous (1)
- 'Actinic Keratoses and Intraepithelial Carcinoma (akiec)' - Cancerous (1)roso (1)


In [ ]:
# Each type of 
label_mapping = {
    "nv": 0,
    "bkl": 1,
    "df": 2,
    "mel": 3,
    "vasc": 4,
    "bcc": 5,
    "akiec": 6
}

In [ ]:
label_mapping2 = {
    "nv": 0,
    "bkl": 0,
    "df": 0,
    "mel": 1,
    "vasc": 0,
    "bcc": 1,
    "akiec": 1
}

## Image Loading

In [ ]:
# function to load_image data

def load_image(image_id  , image_folder):

    image_path = os.path.join(image_folder , f'{image_id}.jpg')

    return Image.open(image_path)

In [ ]:
if path_dir_option == 'local':
    image_folder1 = os.path.join('archive', 'HAM10000_images_part_1')
    image_folder2 = os.path.join('archive', 'HAM10000_images_part_2')

else:
    image_folder1 = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
    image_folder2 = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2'


image_data = []
labels1 = []
labels2 = []


for idx,row in data_hmnist.iterrows():
    i_id = row['image_id']
    dx   = row['dx']
    
    try:
        image = load_image(i_id , image_folder1)

        
    except FileNotFoundError:
        image = load_image(i_id , image_folder2)

    image = image.resize((128,128))
    image = img_to_array(image) /255.0

    image_data.append(image)
    labels1.append(label_mapping[dx])
    labels2.append(label_mapping2[dx])

In [ ]:
classes = ['Melanocytic Nevi (nv)',

           'Benign Keratosis-like Lesions (bkl)',

           'Dermatofibroma (df)',

           'Melanoma (mel)',

           'Vascular Lesions (vasc)',

           'Basal Cell Carcinoma (bcc)',

           'Actinic Keratoses and Intraepithelial Carcinoma (akiec)']

In [ ]:
classes2 = ['Non Cancerous',

            'Cancerous']

Let's display some images from the dataset. 

In [ ]:
def show_samples(i_dex):
    plt.imshow(image_data[i_dex])
    plt.xlabel(f'{classes[labels1[i_dex]]} - {classes2[labels2[i_dex]]}')
    plt.show()

In [ ]:
show_samples(1746)

In [ ]:
plt.figure(figsize=(20,8))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(f'{classes[labels1[i]]} - {classes2[labels2[i]]}')
    plt.imshow(image_data[i])

We have a total of 10,015 images.

In [ ]:
print(len(image_data))
print(len(labels1))

## Data Augmentation
Data augmentation will be applied to increase the number of images and improve model performance..

First, we define the image generator with data augmentation.
In this case, augmentation is applied only to training images with rotations and horizontal/vertical flips..

In [ ]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.4),
])

result = data_augmentation(image_data[10])

plt.figure(figsize=(10, 5))

# Original image
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(image_data[10])

# Augmented image
plt.subplot(1, 2, 2)
plt.title("Augmented Image")
plt.imshow(result)

plt.show()

# Creation of Training and Test Data

Convert labels to numpy array and apply one hot encoding.

In [ ]:
y = np.array(labels1)
from tensorflow.keras.utils import to_categorical
y_encoded = to_categorical(y,num_classes=7)
y_encoded[3]
y_encoded.shape

In [ ]:
X_image = np.array(image_data)
X_image.shape

Split the data into training and test sets, with 80% for training and 20% for testing.

In [ ]:
X_train_image, X_test_image, y_encoded_train ,y_encoded_test = train_test_split(X_image, y_encoded ,test_size=.2 ,random_state=42)

Finally, we apply data augmentation to the training images.


In [ ]:
#aumento de datos de datos prueba
print('Antes del aumento de datos')
print(X_train_image.shape)
print(y_encoded_train.shape)

X_train_augmented = []

y_encoded_train_augmented = []
#new data
for i in range(5000):
    modified_image =  data_augmentation(X_train_image[i])
    X_train_augmented.append(modified_image)
    y_encoded_train_augmented.append(y_encoded_train[i])

#rewrite the original 
for i in range(len(X_train_image)):
    X_train_augmented.append(X_train_image[i])
    y_encoded_train_augmented.append(y_encoded_train[i])
    

X_train_image =  np.array(X_train_augmented)
y_encoded_train = np.array(y_encoded_train_augmented)


print('Despues del aumento de datos')

print(X_train_image.shape)
print(y_encoded_train.shape)

Number of training and test samples

In [ ]:
print(X_train_image.shape)
print(y_encoded_train.shape)

print(X_test_image.shape)
print(y_encoded_test.shape)

# Convolutional Neural Network Model

In [ ]:
# creating a CNN + ANN model

model = Sequential()

model.add(Conv2D(32, kernel_size=(3,3), padding='valid', activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))
model.add(Dropout(0.5))
model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(32, activation='relu'))

# o/p layer
model.add(Dense(7, activation='softmax'))

Model compilation.

In [ ]:

from tensorflow.keras import metrics

# Model compilation with additional metrics
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall')
    ]
)
model.summary()

Model training.

In [ ]:
history = model.fit(X_train_image, y_encoded_train,
                    validation_data=(X_test_image, y_encoded_test),
                    batch_size=32,
                    validation_split=.2,
                    epochs=1)

# Model Evaluation

Below is the loss function and model accuracy on the test dataset.

In [ ]:
# View loss function
plt.xlabel("# Epoca")
plt.ylabel("Magnitud de pérdida")
plt.plot(history.history["loss"])

Muestra de predicciones

In [ ]:
plt.figure(figsize=(20, 10))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(X_test_image[i])
    true_label = classes[np.argmax(y_encoded_test[i])]
    pred_label = classes[np.argmax(model.predict(X_test_image[i].reshape(1, 128, 128, 3)))]
    plt.xlabel(f'True: {true_label}\nPred: {pred_label}')
plt.show()

The model's accuracy on the test images is taken as the percentage of correct classifications.

In [ ]:
model.evaluate(X_test_image, y_encoded_test)


# Additional Metric: F1-score
To further evaluate the model, we can calculate the F1-score, which combines precision and recall into a single metric.

In [ ]:
from sklearn.metrics import f1_score

# Get predictions for the test set
y_pred = model.predict(X_test_image)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_encoded_test, axis=1)

# Calculate F1-score (macro for multiclass)
f1 = f1_score(y_true_classes, y_pred_classes, average='macro')
print(f"F1-score (macro): {f1:.4f}")

# Confusion Matrix
To complement accuracy, precision, recall, and F1-score, we can visualize a confusion matrix for the test set.

- Each **row** corresponds to the *true* class.
- Each **column** corresponds to the *predicted* class.
- Values on the **diagonal** are correct predictions.
- Values **off the diagonal** are misclassifications.

For this multiclass skin cancer problem, a good model will have high values on the diagonal (correctly classified lesions) and low values elsewhere (few confusions between classes).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predictions for the test set
y_pred = model.predict(X_test_image)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_encoded_test, axis=1)

# Compute and plot confusion matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix - Skin Cancer Classes")
plt.show()

# Saving the model in Kaggle for later use in a web page.

In [ ]:
# Save the model

model.save("/kaggle/working/model.h5")

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("/kaggle/working/model.h5")

In [ ]:
model.export("/kaggle/working/saved_model")

Saving in tflite format

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(
    "/kaggle/working/saved_model"
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

tflite_model = converter.convert()

In [ ]:
with open("/kaggle/working/model.tflite", "wb") as f:
    f.write(tflite_model)

Saving in onnx format

In [ ]:
!pip install tflite2onnx

In [ ]:
from tflite2onnx import convert
 
tflite_path = "/kaggle/working/model.tflite"
onnx_path = "/kaggle/working/model.onnx"
 
convert(tflite_path, onnx_path)